# Assignment 2, Hint 2: train long, with a schedule

CSCI 6379 · companion to the [Assignment 2 hints page](https://dongchul.kim/csci6379-fall2026/assignment/2/hints).

**200 epochs**, a peak learning rate of **`1e-3`**, **AdamW**, and a **one-cycle cosine** schedule. The rate starts near zero, climbs to `1e-3` over the first 5% of the steps (warm-up: big random steps at the very start can wreck a fresh network), then glides down along a cosine to almost zero by the last step. Big steps early explore; small steps late settle into the bottom of the valley. With Hint 4's augmentation every epoch looks different, so the network needs many of them.

**Where it goes.** Two places in your notebook: the settings at the top of `recipe.py`, and the two lines in `train_loop.py` that create `opt` and `sched`. The loop already calls `sched.step()` once per batch and saves the scheduler in its checkpoints, so nothing else changes.

This notebook holds **only the code of this hint**. It is not a training program and it
does not touch the assignment data. Copy the marked cell into your own Assignment 2
notebook, change `RUN_NAME` there, and train as before. The last cell is a small check
you can run here to see the hint working.

## Hint 2, part 1: copy into the settings at the top of `recipe.py`

In [ ]:
# ---- recipe.py: change these three settings ----
EPOCHS   = 200            # was 10 (FIX 1 made it 40). Long, because augmentation makes every epoch different.
LR       = 1e-3           # the PEAK learning rate; the schedule below decides the rate at each step
RUN_NAME = "unet4-hints"  # a new name: the schedule's length is fixed when a run starts, so a run
                          # with a different EPOCHS must not resume an old checkpoint

## Setup for this notebook only (do not copy)

A stand-in network and the number of steps that `train_loop.py` computes (800 training images, batch 16: 50 steps an epoch, 10,000 in 200 epochs).

In [ ]:
# setup for this notebook only (do not copy): a stand-in network and the numbers train_loop.py computes
import torch, torch.nn as nn
import matplotlib.pyplot as plt
net = nn.Linear(4, 1)                      # any network will do for looking at the schedule
BS, N_TRAIN = 16, 800       # EPOCHS and LR come from the part 1 cell above
steps = EPOCHS * ((N_TRAIN + BS - 1) // BS)
print("optimizer steps in the run:", steps)

## Hint 2, part 2: in `train_loop.py`, replace the two lines that create `opt` and `sched`

In [ ]:
# ---- train_loop.py: these two statements replace "opt = torch.optim.Adam(...)" and "sched = None" ----
# (net, LR and steps already exist at this point in train_loop.py; the setup cell below fakes them here)

# AdamW = Adam with "decoupled" weight decay: every step also shrinks each weight by a tiny
# fraction (LR * 1e-4 of itself), which keeps the weights small and the network less eager to
# memorise. It is not the same as Adam's own weight_decay argument, which interacts with Adam's scaling.
opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)

# One-cycle schedule over the whole run:
#   total_steps   = steps, the number of optimizer steps in the run (EPOCHS x batches per epoch),
#                   already computed in train_loop.py. Stepping past it raises an error.
#   pct_start     = 0.05: the first 5% of the steps warm the rate up from LR/25 to LR
#   anneal_strategy="cos": then it falls along half a cosine wave to almost zero (LR / 250,000) at the end
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps,
                                            pct_start=0.05, anneal_strategy="cos")

## Check (run here)

Steps through the whole schedule and plots the learning rate the scheduler hands out at every step.

In [ ]:
# plot the learning rate at every step (run the part 2 cell first)
rates = []
for _ in range(steps):
    rates.append(sched.get_last_lr()[0])  # the rate the next step will use
    opt.step()                             # (nothing to learn here; the scheduler only needs a step to follow)
    sched.step()                           # once per batch, as in train_loop.py
plt.figure(figsize=(7, 3))
plt.plot([s / (steps / EPOCHS) for s in range(steps)], rates)
plt.xlabel("epoch"); plt.ylabel("learning rate"); plt.title("one-cycle cosine, peak 1e-3")
plt.show()
print(f"start {rates[0]:.1e}, peak {max(rates):.1e} at epoch {rates.index(max(rates)) / 50:.0f}, end {rates[-1]:.1e}")